In [ ]:
import pandas as pd

In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
# Load market data
market = pd.read_csv("../../data/processed/market_all_features.csv", parse_dates=["date"])
market["ticker"] = market["ticker"].astype(str)

print("Market shape:", market.shape)
display(market.head())


In [ ]:
# Load sentiment (Parquet is preferred)
sent = pd.read_parquet("../../data/processed/sentiment_daily.parquet")
sent["date"] = pd.to_datetime(sent["date"])
sent["ticker"] = sent["ticker"].astype(str)

print("Sentiment shape:", sent.shape)
display(sent.head())


In [ ]:
# Merge
features = market.merge(
    sent,
    on=["ticker", "date"],
    how="left",    # keep ALL market rows (important!)
    validate="many_to_one"  # each (ticker, date) in sentiment appears once
)

In [ ]:
print("Merged shape:", features.shape)
features.head()

In [ ]:
# 1. Sort by ticker/date (always good for time series)
features = features.sort_values(["ticker", "date"]).reset_index(drop=True)

# 2. Drop rows where return is NaN (first day per ticker)
before = features.shape[0]
features = features[~features["return"].isna()].copy()
after = features.shape[0]
print(f"Dropped {before - after} rows with NaN return.")

# 3. Create stable row_id
features["row_id"] = features["ticker"] + "|" + features["date"].astype(str)

# 4. Fill sentiment NaNs for no-news days
sent_cols_num = [
    "sent_mean", "sent_std", "sent_count",
    "sent_pos", "sent_neg", "sent_neu",
    "sent_lag1", "sent_roll3",
]

for col in sent_cols_num:
    if col in features.columns:
        features[col] = features[col].fillna(0)

if "sent_daily_cat" in features.columns:
    features["sent_daily_cat"] = features["sent_daily_cat"].fillna("neutral")

features.head()


In [ ]:
from sklearn.model_selection import TimeSeriesSplit
import numpy as np

# 1) Ensure sorted by time (and ticker) for deterministic order
features = features.sort_values(["date", "ticker"]).reset_index(drop=True)

# 2) Prepare TimeSeriesSplit
n_splits = 5  # you can change this if you want more/fewer folds
tscv = TimeSeriesSplit(n_splits=n_splits)

# 3) Take the LAST split as final train/test
all_splits = list(tscv.split(features))
train_idx, test_idx = all_splits[-1]

print(f"Train size: {len(train_idx)}, Test size: {len(test_idx)}")

# 4) Build is_train mask
is_train = np.zeros(len(features), dtype=bool)
is_train[train_idx] = True

# 5) Create split DataFrame
split = features[["row_id", "date"]].copy()
split["is_train"] = is_train

print(split["is_train"].value_counts())
split.head()


In [ ]:
split.to_csv("../../data/processed/split_v1.csv", index=False)



In [ ]:
features.to_csv("../../data/processed/features.csv", index=False)
print("Saved ../../data/processed/features.csv")


In [ ]:
features.to_parquet("../../data/processed/features.parquet", index=False)


In [ ]:
features.shape